# API Data Freshness Check

Most recent available timestamp for each training data source, with lag in hours.

In [6]:
import io, zipfile
import requests
import pandas as pd

NOW = pd.Timestamp.now()
print(f"Now: {NOW.strftime('%Y-%m-%d %H:%M')}")

Now: 2026-03-29 09:17


In [7]:
# Open-Meteo — ERA5 archive, typically ~5-day lag; hourly resolution
r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 40.7128, "longitude": -74.0060,
    "start_date": (NOW - pd.Timedelta(days=14)).strftime("%Y-%m-%d"),
    "end_date":   NOW.strftime("%Y-%m-%d"),
    "hourly": "temperature_2m",
    "timezone": "America/New_York",
})
r.raise_for_status()
h = r.json()["hourly"]
temps = pd.Series(h["temperature_2m"], index=pd.to_datetime(h["time"]))
latest_openmeteo = temps.last_valid_index()
lag_h = (NOW - latest_openmeteo).total_seconds() / 3600
print(f"Open-Meteo:  {latest_openmeteo}  ({lag_h:.0f}h lag)")

Open-Meteo:  2026-03-29 23:00:00  (-14h lag)


In [8]:
# NYISO — 5-min interval data; walk back up to 3 months until a zip is available
latest_nyiso = None
for p in pd.period_range(end=NOW, periods=3, freq="M")[::-1]:
    r = requests.get(
        f"https://mis.nyiso.com/public/csv/pal/{p.year}{p.month:02d}01pal_csv.zip",
        timeout=15,
    )
    if r.ok:
        with zipfile.ZipFile(io.BytesIO(r.content)) as z:
            raw = pd.concat([pd.read_csv(z.open(n)) for n in z.namelist()])
        latest_nyiso = pd.to_datetime(raw["Time Stamp"]).max()
        lag_h = (NOW - latest_nyiso).total_seconds() / 3600
        print(f"NYISO:       {latest_nyiso}  ({lag_h:.0f}h lag)")
        break

NYISO:       2026-03-29 09:00:00  (0h lag)


In [9]:
# MTA daily ridership — daily cadence so latest timestamp is midnight of most recent date
r = requests.get("https://data.ny.gov/resource/sayj-mze2.json",
                 params={"$order": "date DESC", "$limit": "1"}, timeout=30)
r.raise_for_status()
latest_mta = pd.to_datetime(r.json()[0]["date"])
lag_h = (NOW - latest_mta).total_seconds() / 3600
print(f"MTA:         {latest_mta}  ({lag_h:.0f}h lag)")

# NYC 311 — sub-hour timestamp precision
r = requests.get("https://data.cityofnewyork.us/resource/erm2-nwe9.json",
                 params={"$order": "created_date DESC", "$limit": "1"}, timeout=30)
r.raise_for_status()
latest_311 = pd.to_datetime(r.json()[0]["created_date"])
lag_h = (NOW - latest_311).total_seconds() / 3600
print(f"311:         {latest_311}  ({lag_h:.0f}h lag)")

MTA:         2026-03-26 00:00:00  (81h lag)
311:         2026-03-28 02:54:36  (30h lag)


In [10]:
summary = pd.DataFrame.from_dict({
    "Open-Meteo (ERA5)": latest_openmeteo,
    "NYISO Zone J":      latest_nyiso,
    "MTA Ridership":     latest_mta,
    "NYC 311":           latest_311,
}, orient="index", columns=["latest_timestamp"])
summary["lag_hours"] = summary["latest_timestamp"].apply(
    lambda t: round((NOW - t).total_seconds() / 3600, 1)
)
summary.index.name = "source"
summary

,latest_timestamp,lag_hours
source,,
Open-Meteo (ERA5),2026-03-29 23:00:00,-13.7
NYISO Zone J,2026-03-29 09:00:00,0.3
MTA Ridership,2026-03-26 00:00:00,81.3
NYC 311,2026-03-28 02:54:36,30.4
